# Deploying a Scalable ML Pipeline with FastAPI
## Initial data exploration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
data = pd.read_csv("data/census.csv")

In [2]:
# Display a sample of the data
display(data.head())

,age,workclass,fnlgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,salary
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [3]:
# Inspect dimensions
print(f'Rows: {data.shape[0]:,}')
print(f'Rows: {data.shape[1]}')

Rows: 32,561
Rows: 15


In [4]:
# Inspect column names and data types
print(data.info())

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       32561 non-null  str  
 2   fnlgt           32561 non-null  int64
 3   education       32561 non-null  str  
 4   education-num   32561 non-null  int64
 5   marital-status  32561 non-null  str  
 6   occupation      32561 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital-gain    32561 non-null  int64
 11  capital-loss    32561 non-null  int64
 12  hours-per-week  32561 non-null  int64
 13  native-country  32561 non-null  str  
 14  salary          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB
None


In [5]:
# Summary statistics
display(data.describe(include='all').T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
age,32561.0,NaN,NaN,NaN,38.581647,13.640433,17.0,28.0,37.0,48.0,90.0
workclass,32561,9,Private,22696,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fnlgt,32561.0,NaN,NaN,NaN,189778.366512,105549.977697,12285.0,117827.0,178356.0,237051.0,1484705.0
education,32561,16,HS-grad,10501,NaN,NaN,NaN,NaN,NaN,NaN,NaN
education-num,32561.0,NaN,NaN,NaN,10.080679,2.57272,1.0,9.0,10.0,12.0,16.0
marital-status,32561,7,Married-civ-spouse,14976,NaN,NaN,NaN,NaN,NaN,NaN,NaN
occupation,32561,15,Prof-specialty,4140,NaN,NaN,NaN,NaN,NaN,NaN,NaN
relationship,32561,6,Husband,13193,NaN,NaN,NaN,NaN,NaN,NaN,NaN
race,32561,5,White,27816,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sex,32561,2,Male,21790,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Count standard missing values
print("NaN values per column")
display(data.isna().sum().sort_values(ascending=False))

NaN values per column


age               0
workclass         0
fnlgt             0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
salary            0
dtype: int64

In [9]:
# Count qustion marks and surrounding whitespace
categorical_columns = data.select_dtypes(include=['str']).columns

for column in categorical_columns:
    count = (
        data[column].astype('string').str.strip().eq('?').sum()
    )
    if count > 0:
        print(f'{column}: {count} "?" values')

workclass: 1836 "?" values
occupation: 1843 "?" values
native-country: 583 "?" values


In [8]:
# Inspect whitespace in column names
print([repr(column) for column in data.columns])

["'age'", "'workclass'", "'fnlgt'", "'education'", "'education-num'", "'marital-status'", "'occupation'", "'relationship'", "'race'", "'sex'", "'capital-gain'", "'capital-loss'", "'hours-per-week'", "'native-country'", "'salary'"]


In [10]:
missing_mask = data.select_dtypes(include=['str']).eq('?')

rows_with_missing = missing_mask.any(axis=1).sum()

print(f"Rows with at least one '?': {rows_with_missing:,}")
print(f"Percentage of rows: {rows_with_missing / len(data) * 100:.2f}")

Rows with at least one '?': 2,399
Percentage of rows: 7.37


In [11]:
def income_by_category(data, column):
    result = pd.crosstab(
        data[column],
        data['salary'],
        normalize='index'
    )*100

    result['count'] = data[column].value_counts()

    return result.sort_values( by='>50K', ascending=False).round(2)

In [12]:
display(income_by_category(data, 'education')) 
display(income_by_category(data, 'sex')) 
display(income_by_category(data, 'race')) 
display(income_by_category(data, 'workclass')) 

salary,<=50K,>50K,count
education,,,
Doctorate,25.91,74.09,413
Prof-school,26.56,73.44,576
Masters,44.34,55.66,1723
Bachelors,58.52,41.48,5355
Assoc-voc,73.88,26.12,1382
Assoc-acdm,75.16,24.84,1067
Some-college,80.98,19.02,7291
HS-grad,84.05,15.95,10501
12th,92.38,7.62,433


salary,<=50K,>50K,count
sex,,,
Male,69.43,30.57,21790
Female,89.05,10.95,10771


salary,<=50K,>50K,count
race,,,
Asian-Pac-Islander,73.44,26.56,1039
White,74.41,25.59,27816
Black,87.61,12.39,3124
Amer-Indian-Eskimo,88.42,11.58,311
Other,90.77,9.23,271


salary,<=50K,>50K,count
workclass,,,
Self-emp-inc,44.27,55.73,1116
Federal-gov,61.35,38.65,960
Local-gov,70.52,29.48,2093
Self-emp-not-inc,71.51,28.49,2541
State-gov,72.80,27.20,1298
Private,78.13,21.87,22696
?,89.60,10.40,1836
Never-worked,100.00,0.00,7
Without-pay,100.00,0.00,14
